# PF4 Heart Failure Associations from HERMES GWAS Including UK Biobank Samples

This notebook processes heart failure summary statistics from the HERMES dataset including UK Biobank samples to identify SNP associations within the PF4 genomic region.

In [1]:
import requests
import pandas as pd
import json
from pathlib import Path

pd.set_option("display.max_columns", None)

In [2]:
region_path = Path("../data/interim/ensembl/region.json")

hermes_path = Path("../data/raw/hermes/hermes_hf_ukb.txt")

out_dir = Path("../data/interim/hermes")
out_dir.mkdir(parents=True, exist_ok=True)

out_associations_csv = out_dir / "ukb_associations.csv"

In [3]:
with open(region_path, "r") as f:
    pf4_region = json.load(f)

print("Gene:", pf4_region["gene_symbol"])
print("Assembly:", pf4_region["assembly_name"])
print(
    "PF4 region:",
    f"chr{pf4_region['chromosome']}:{pf4_region['region_start']}-{pf4_region['region_end']}"
)

Gene: PF4
Assembly: GRCh38
PF4 region: chr4:73930811-74032027


In [4]:
hermes_preview = pd.read_csv(
    hermes_path,
    sep="\t",
    nrows=5
)

hermes_preview

,SNP,CHR,BP,A1,A2,freq,b,se,p,N
0,rs1983867,10,100016196,c,g,0.4389,-0.0231,0.0078,0.003234,964057
1,rs2296438,10,100146895,t,c,0.6544,0.0024,0.0082,0.769400,964057
2,rs2182168,10,100148353,c,g,0.6545,0.0023,0.0082,0.777700,964057
3,rs10883085,10,100155613,a,t,0.6890,0.0055,0.0084,0.511800,964057
4,rs7907555,10,100155810,t,c,0.3099,-0.0048,0.0085,0.573400,964057


In [5]:
# HERMES full summary statistics use hg19 / GRCh37 coordinates.
# Therefore, we retrieve the PF4 region from Ensembl GRCh37
# and keep the HERMES coordinates explicitly as GRCh37.

hermes_assembly = "GRCh37"
flanking_bp = int(pf4_region["flanking_bp"])

server = "https://grch37.rest.ensembl.org"
endpoint = f"/lookup/symbol/homo_sapiens/{pf4_region['gene_symbol']}"

response = requests.get(
    server + endpoint,
    headers={
        "Content-Type": "application/json",
        "Accept": "application/json"
    },
    timeout=30
)

if not response.ok:
    response.raise_for_status()

pf4_grch37 = response.json()

hermes_region = {
    "gene_symbol": pf4_region["gene_symbol"],
    "chromosome": str(pf4_grch37["seq_region_name"]),
    "gene_start": int(pf4_grch37["start"]),
    "gene_end": int(pf4_grch37["end"]),
    "region_start": max(1, int(pf4_grch37["start"]) - flanking_bp),
    "region_end": int(pf4_grch37["end"]) + flanking_bp,
    "assembly_name": "GRCh37",
    "flanking_bp": flanking_bp,
}

print("HERMES full UKB filtering region:")
print(
    f"chr{hermes_region['chromosome']}:"
    f"{hermes_region['region_start']}-{hermes_region['region_end']}"
)
print("Assembly:", hermes_region["assembly_name"])

HERMES full UKB filtering region:
chr4:74796794-74897841
Assembly: GRCh37


In [6]:
usecols = [
    "SNP",
    "CHR",
    "BP",
    "A1",
    "A2",
    "freq",
    "b",
    "se",
    "p",
    "N"
]

matched_chunks = []

region_chromosome = str(hermes_region["chromosome"]).replace("chr", "")
region_start = int(hermes_region["region_start"])
region_end = int(hermes_region["region_end"])

for chunk in pd.read_csv(
    hermes_path,
    sep="\t",
    usecols=usecols,
    chunksize=500_000
):
    chunk["CHR_filter"] = (
        chunk["CHR"]
        .astype(str)
        .str.replace("chr", "", regex=False)
    )

    chunk["BP_filter"] = pd.to_numeric(chunk["BP"], errors="coerce")

    matched = chunk[
        (chunk["CHR_filter"] == region_chromosome) &
        (chunk["BP_filter"].between(region_start, region_end))
    ].copy()

    matched = matched.drop(columns=["CHR_filter", "BP_filter"])

    if not matched.empty:
        matched_chunks.append(matched)

if matched_chunks:
    hermes_ukb_matches_df = pd.concat(matched_chunks, ignore_index=True)
else:
    hermes_ukb_matches_df = pd.DataFrame(columns=usecols)

hermes_ukb_matches_df.shape

(191, 10)

In [7]:
hermes_ukb_associations_df = hermes_ukb_matches_df.rename(columns={
    "SNP": "rsID",
    "CHR": "HERMES_GRCh37_CHR",
    "BP": "HERMES_GRCh37_BP",
    "A1": "HERMES_effect_allele",
    "A2": "HERMES_other_allele",
    "freq": "HERMES_effect_allele_frequency",
    "b": "HERMES_effect",
    "se": "HERMES_std_error",
    "p": "HERMES_p_value",
    "N": "HERMES_total_sample_size"
})

hermes_ukb_associations_df.head()

,rsID,HERMES_GRCh37_CHR,HERMES_GRCh37_BP,HERMES_effect_allele,HERMES_other_allele,HERMES_effect_allele_frequency,HERMES_effect,HERMES_std_error,HERMES_p_value,HERMES_total_sample_size
0,rs352023,4,74797946,a,t,0.8823,-0.0057,0.0121,0.6385,960991.0
1,rs164616,4,74798566,t,c,0.1178,0.0062,0.0121,0.6064,960991.0
2,rs164615,4,74801525,a,g,0.1176,0.0061,0.0121,0.6146,960991.0
3,rs4694660,4,74840445,a,t,0.8239,-0.0057,0.0103,0.5765,960981.0
4,rs8180167,4,74833373,a,t,0.1764,0.0062,0.0102,0.5455,960976.0


In [8]:
numeric_cols = [
    "HERMES_GRCh37_CHR",
    "HERMES_GRCh37_BP",
    "HERMES_effect_allele_frequency",
    "HERMES_effect",
    "HERMES_std_error",
    "HERMES_p_value",
    "HERMES_total_sample_size"
]

for col in numeric_cols:
    hermes_ukb_associations_df[col] = pd.to_numeric(
        hermes_ukb_associations_df[col],
        errors="coerce"
    )

hermes_ukb_associations_df = hermes_ukb_associations_df.sort_values(
    "HERMES_p_value",
    ascending=True,
    na_position="last"
)

hermes_ukb_associations_df.head()

,rsID,HERMES_GRCh37_CHR,HERMES_GRCh37_BP,HERMES_effect_allele,HERMES_other_allele,HERMES_effect_allele_frequency,HERMES_effect,HERMES_std_error,HERMES_p_value,HERMES_total_sample_size
141,rs111720070,4,74869496,a,g,0.0244,0.0492,0.0327,0.1327,899406.0
144,rs115183538,4,74805105,a,g,0.0218,-0.0471,0.0329,0.1525,889680.0
157,rs13120504,4,74825489,a,c,0.8210,-0.0160,0.0122,0.1895,565725.0
153,rs4694657,4,74822074,a,t,0.8235,-0.0155,0.0119,0.1926,591259.0
152,rs17814280,4,74895277,t,c,0.0202,0.0414,0.0347,0.2334,867372.0


In [9]:
rows_before_deduplication = len(hermes_ukb_associations_df)

hermes_ukb_associations_df = hermes_ukb_associations_df.drop_duplicates()

rows_after_deduplication = len(hermes_ukb_associations_df)
duplicate_rows_removed = rows_before_deduplication - rows_after_deduplication

duplicate_rows_removed

0

In [10]:
hermes_ukb_associations_df.to_csv(out_associations_csv, index=False)

print("Saved:", out_associations_csv)

Saved: ../data/interim/hermes/ukb_associations.csv


In [11]:
summary = {
    "source_dataset": "HERMES Heart Failure summary statistics including UK Biobank samples",
    "phenotype": "Heart failure",
    "region": (
        f"chr{hermes_region['chromosome']}:"
        f"{hermes_region['region_start']}-{hermes_region['region_end']}"
    ),
    "region_assembly": hermes_region["assembly_name"],
    "associations": int(len(hermes_ukb_associations_df)),
    "unique_rsIDs": int(hermes_ukb_associations_df["rsID"].nunique(dropna=True)),
    "duplicate_rows_removed": int(duplicate_rows_removed),
}

summary

{'source_dataset': 'HERMES Heart Failure summary statistics including UK Biobank samples',
 'phenotype': 'Heart failure',
 'region': 'chr4:74796794-74897841',
 'region_assembly': 'GRCh37',
 'associations': 191,
 'unique_rsIDs': 191,
 'duplicate_rows_removed': 0}